# 15-зертханалық жұмыс. Қорытынды жоба: GK жіктеуін әділ бағалау

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 1-15-дәрістер; бастапқы Python, pandas, NumPy және статистика.

## Оқу мақсаттары

- Деректерді дайындауды, Pipeline мен ағып кетуді бақылауды байланыстыру
- DummyClassifier мен логистикалық регрессияны салыстыру
- Метрикалар мен шектеулері бар тексерілетін есеп ұсыну

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `numpy`, `pandas`, `scikit-learn`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Қорытынды есеп

Тақырып бастапқы 1-14-дәрістер мен зертханалық жұмыстарға қосымша әзірленген. Бұл жоқ 15-бастапқы материалды қалпына келтіру емес, жаңа интеграциялық жоба. Сіз football.csv жолдарының жалпы сандық сипаттамалары бойынша позиция GK деп жазылғанын болжауыңыз керек. 1 класы GK, 0 класы қалған барлық позицияны білдіреді.

Негізгі бағаланатын нәтиже — дұрыс қайталанатын процесс: X/y анықтау, деректер ағып кетпеуі, стратификацияланған бөлу, базалық модель, Pipeline, айқас тексеру тек train ішінде және test бойынша бір реттік қорытынды бағалау. Жоғары F1 сынақтан өтудің міндетті шарты емес. Баптау таңдағанда test-ті қарап нәтижені жақсартуға болмайды.

Файл тарихи, жетіспейтін мәндерсіз 12897 жолы мен 42 бағаны бар. Оның бас жиынтығы мен іріктеу сызбасы белгісіз. Есеп оқу мақсатында берілген: алынған метрикалар футболшыларды нақты таңдаудың немесе қазіргі құрамдарға жалпылаудың сапасын дәлелдемейді. Айқын GK дағдылары шығарылғанның өзінде қалған рейтингтер бастапқы дереккөздің белгілеу рәсімін көрсетуі мүмкін.

## Құралдар және бастапқы мысал

Python, pandas, NumPy және scikit-learn қолданыңыз. Қосымша деректер мен интернет қажет емес. Жинақ түбірінен data/football.csv файлы арқылы жұмыс істеңіз. Барлық нәтиже файлын work_results ішіне орналастырыңыз. Бөлу мен CV үшін random_state=42 қолданыңыз. Өзіңіздің симуляцияңызды қоссаңыз, np.random.default_rng(42) жасаңыз.

Талданған мысал жобадан өзгеше жасанды жауаптарға қатысты. Ол бекітілген [0,1] ретімен қателер матрицасының мағынасын көрсетеді. Сегіз нысанның алтауы дұрыс жіктелген; бір жалған дабыл және оң класты бір өткізіп алу бар. Мұнда дұрыс жауаптар үлесі (accuracy) мен F1 0.75-ке тең. Кластардың басқа құрылымында бұл метрикалар едәуір ерекшеленуі мүмкін.



In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score,
)

out = Path('work_results')
out.mkdir(exist_ok=True)
toy_true = np.array([0, 0, 0, 0, 1, 1, 1, 1])
toy_pred = np.array([0, 0, 0, 1, 0, 1, 1, 1])
print(confusion_matrix(toy_true, toy_pred, labels=[0, 1]))
print(accuracy_score(toy_true, toy_pred))
print(f1_score(toy_true, toy_pred))



Жобада теріс және оң нысандар саны қатты ерекшеленуі мүмкін. Сондықтан дұрыс жауаптар үлесі (accuracy) теңгерілген дұрыс жауаптар үлесімен (balanced accuracy), GK дәлдігімен (precision), GK толықтығымен (recall) және F1 GK-мен бірге беріледі. Оң болжамдар болмағанда zero_division=0 беріп, осы келісімді түсіндіріңіз. Матрица белгілерінде тек төрт сан емес, шынайы және болжанған класс болуы керек.

## Белгілер келісімі және міндетті бөлу

Тек алдын ала таңдалған алты арнайы емес сандық белгіні қолданыңыз. Төменде оқу мен бөлудің толық бастамасы берілген. Бұл модельдеудің шешімі емес: базалық модель, Pipeline, CV, қорытынды бағалау және есеп сіздің жұмысыңыз болып қалады.



In [ ]:
football = pd.read_csv(Path('data') / 'football.csv')
features = [
    'Age', 'SprintSpeed', 'Stamina',
    'Strength', 'Reactions', 'ShortPassing',
]
X = football.loc[:, features].copy()
y = football['Position'].eq('GK').astype('int64')
assert football.shape == (12897, 42)
assert not X.isna().any().any()
assert not any(c.startswith('GK') for c in X.columns)
assert 'Position' not in X.columns
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in X)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(football)
print('Train:', X_train.shape, 'Test:', X_test.shape)



X ішінде тыйым салынғандар: Position, GK префиксі бар барлық баған, Name, Club, Nationality, Wage, Value және Unnamed: 0. Position жауапты тікелей береді. GK көрсеткіштері рөл белгісіне тым жақын және жаттығу мағынасын өзгертеді. Атау сенімді бірегей идентификатор емес, ескі индекс мазмұнды белгі емес. Penalties — гол саны емес, пенальти орындау дағдысы; ол рұқсат етілген жиынға кірмейді.

Қорытынды кезеңге дейін test метрикаларын есептемеңіз және test қателерін зерттемеңіз. Индекстердің қиылыспауын және бөліктер көлемін тексеруге болады: бұл модель іріктеу емес, бөлуді тексеру. Шешім қабылдау үшін белгілер үлестірімі мен класс теңгерімін train ішінде қарастырыңыз. Сәттірек нәтиже үшін басқа seed таңдамаңыз.

## Тапсырмалар

1. **Есеп паспорты және схема аудиті.** Бақылау бірлігін, X, y, оң класты және белгілерді қолданудың болжанған сәтін сипаттаңыз. Алты белгінің барын, сандық типтерін және жетіспейтін мәндер жоқтығын тексеріңіз. Жақсы нәтиже берсе де, Position мен GK дағдыларының неге шығарылғанын бөлек түсіндіріңіз. Тарихи файл шектеулерін және Wage/Value бірліктері расталмағанын көрсетіңіз. Python, pandas, NumPy және scikit-learn техникалық нұсқаларын есепке не JSON паспортқа сақтаңыз.

2. **Бөлуді тексеру.** Берілген test_size=0.2, random_state=42, stratify=y бөлуді қолданыңыз. Бастапқы индекстердің қиылыспауын және жалпы жол санының сақталуын растаңыз. Train ішінде класс сандары мен үлестерін есептеңіз. Стратификация неге қажет екенін және нені қамтамасыз етпейтінін түсіндіріңіз: жолдар тәуелсіздігін, басқа клубтарға жалпылауды және қайталама мәндік нысандардың жоқтығын. Name-ді бірегейлік дәлелі ретінде қолданбаңыз.

3. **Базалық ереже.** DummyClassifier(strategy='most_frequent') жасаңыз. Test шақыруына дейін GK train-дегі азшылық болса, осы ереженің GK дәлдігі/толықтығы бойынша қалай әрекет ететінін болжаңыз. Базалық модельді негізгі модельмен бірдей ішкі CV бөлінулерінде бағалаңыз. Базалық модель нәтижелері кестені формалды толтыруға емес, пайданы түсінуге қажет.

4. **Pipeline ішіндегі негізгі модель.** StandardScaler және LogisticRegression(C=1.0, max_iter=1000, random_state=42) құралдарынан Pipeline жасаңыз. Split-ке дейін толық X ішінде fit_transform шақырмаңыз және CV алдында бүкіл train-ді масштабтамаңыз. Scaler қандай шамаларды үйренетінін және әр CV қайталауында оларды неліктен жаңадан бағалау керектігін түсіндіріңіз. Алдын ала көрсетілген параметрлерді қолданыңыз; бұл жобада C баптау және белгілерді кеңейту қажет емес.

5. **Тек train ішіндегі айқас тексеру.** StratifiedKFold(n_splits=5, shuffle=True, random_state=42) жасаңыз. Базалық модель мен Pipeline үшін X_train, y_train ішінде balanced_accuracy және f1 арқылы cross_validate орындаңыз. Әр fold мәнін, орташа мән мен стандартты ауытқуды сақтаңыз. Есепте cross_validate ішіндегі test_f1/test_balanced_accuracy кілттері train-нің ішкі тексеру бөліктеріне қатысты екенін нақтылаңыз. Бұл шақыруға X_test/y_test берілмейді. Бес бағаның шашырауын автоматты 95% сенімділік аралығы деп жарияламаңыз.

6. **Test алдында шешімді бекіту.** Қорытынды бағалауға дейін lab15_plan.json файлын жасаңыз. Белгілер тізімін, мақсат анықтамасын, split параметрлерін, екі алгоритмді, CV параметрлерін, қорытынды метрикалар жиынын және логистикалық модель үшін бекітілген 0.5 шегін жазыңыз. CV нәтижелерінен не шешкеніңізді және қандай шектеулер тапқаныңызды қысқаша түсіндіріңіз. Жоспар жасалғаннан кейін test нәтижелері бойынша жоба параметрлерін өзгертпеңіз. Кез келген алгоритмнің жеңісі емес, рәсімді сақтау бағаланады.

7. **Бір реттік қорытынды тексеру.** Жоспар бекітілгеннен кейін алдын ала таңдалған екі estimator-ды бүкіл train ішінде оқытыңыз. Бір қорытынды кезеңде әрқайсысының X_test болжамдарын алып, accuracy, balanced accuracy, precision GK, recall GK, F1 GK және labels=[0,1] болатын confusion matrix есептеңіз. Болжамдар мен метрикаларды сақтаңыз. Нәтижені көргеннен кейін жаңа split жасамаңыз, C, шек не белгілерді өзгертпеңіз. Жақсарту идеясы пайда болса, алынған бағаны сақтап, оны келесі тәжірибеге ұсыныс ретінде рәсімдеңіз.

8. **Қателерді талдау және мазмұнды есеп.** FP және FN болса, жолдың техникалық индексімен және алты белгімен әр түрдің үшке дейін жолын көрсетіңіз. Қатенің бір түрі жоқ болса, солай жазыңыз. Талдау қорытынды тексеруден кейін сипаттамалық түрде орындалып, қайта баптауға қолданылмайды. Барлық метрика бойынша базалық модель мен Pipeline-ды салыстырыңыз. Жоғарырақ дұрыс жауаптар үлесін сирек GK үшін пайданың жеткілікті дәлелі деп атамаңыз. Белгілер мен рөлдің байланысы себептілікті дәлелдемейтінін көрсетіңіз.

9. **Жалпылау шектері және есептерге шолу.** Қорытындыда жолдарды кездейсоқ бөлу жаңа клубтарға немесе жылдарға жалпылауды неліктен тексермейтінін түсіндіріңіз. Тұтас клубтар оқытудан тыс қалатын және бөлек жаңа деректер қолданылатын келесі дизайнды ұсыныңыз. Футбол сипаттамалары бойынша бір ықтимал регрессия сұрағын және бір кластерлеу сұрағын қысқаша келтіріңіз; әрқайсысының жауап типін және бағалау тәсілін атаңыз. Осы жобаның бөлек сақталған test деректерінде қосымша модельдер оқытпаңыз.

## Файлдар және күтілетін тексерулер

Notebook не скрипт, есеп және work_results ішіндегі бес файлды тапсырыңыз: lab15_plan.json, lab15_cv.csv, lab15_metrics.csv, lab15_predictions.csv және lab15_confusion.csv. Есеп қойылымды, деректер паспортын, ағып кетуге қарсы дәлелдерді, train/CV/test процесін, модельдер салыстыруын және шектеулерді қамтиды. Барлық бөлік таза күйден жоғарыдан төменге дейін орындалуы керек.

| Файл | Бағандар немесе өрістер | Инварианттар |
|---|---|---|
| lab15_plan.json | features, target, split, models, cv, metrics | Тыйым салынған белгілер жоқ |
| lab15_cv.csv | model, fold, metric, value | 2 модель, 5 бөлік, 2 метрика |
| lab15_metrics.csv | model, accuracy, balanced_accuracy, precision_GK, recall_GK, f1_GK | 2 жол, мәндері 0-ден 1-ге дейін |
| lab15_predictions.csv | row_id, y_true, dummy_pred, logistic_pred | Әр test нысанына бір жол |
| lab15_confusion.csv | model, true_label, pred_label, count | Әр модель үшін 4 үйлесім |

row_id — ағымдағы DataFrame жолының техникалық бастапқы индексі, тек нәтижелерді сәйкестендіруге қажет; бұл спортшының расталған идентификаторы емес. Әр модель үшін төрт count қосындысы test көлеміне тең болуы керек. Диагональ қосындысының test көлеміне бөліндісі accuracy-ге сәйкес келуі керек. Болжам мәндері {0,1} жиынына тиесілі, y_true реті X_test ретіне сәйкес болуы керек.

CV тек train индекстерін қолданатынын тексеріңіз. scaler.mean_ мәнін қолмен көрсетсеңіз, оны бүкіл файлдың орташасымен емес, қорытынды fit-тен кейінгі X_train mean мәнімен салыстырыңыз. Жинақталу warning-і болса, test-ке дейін себебін анықтап бекітіңіз; әдемі есеп үшін ескертуді жасырмаңыз. Көрсетілген max_iter бұл оқу сызбасына әдетте жеткілікті, бірақ орындалуын тексеру керек.

Test бойынша көп рет іріктегеннен кейін алынған сәтті нәтижені жалғыз әділ қорытынды баға ретінде сақтауға болмайды. Рәсім бұзылса, бұзушылықты және жаңа тексеру деректері қажеттігін көрсетіңіз. Шектеуді адал мойындау негізсіз «дұрыс жауаптар үлесі дәлелденді» тұжырымынан жақсы. Test-тің өзі шектеулі ақырлы жиын, басқа жарамды нысандар жиынында оның метрикалары өзгеруі мүмкін.

Қосымша тапсырма тек келесі бөлек тәжірибе ретінде орындалады: клубтар тұтас бір бөлікте қалатындай Club бойынша топтық split жобалаңыз. Клубтардағы GK саны әртүрлі болғанда қандай мәселелер туатынын және топтармен CV қалай ұйымдастырылатынын сипаттаңыз. Жоспар жеткілікті; қаралған test-ті жаңа баптау дереккөзі ретінде қолданбаңыз.

## Қорғауға арналған сұрақтар

1. Бағдарламаңыздың қай жерінде Position және барлық GK көрсеткіштері шығарылған?
2. StandardScaler нені және әр CV қайталауында қандай жолдардан үйренеді?
3. Базалық модельде дұрыс жауаптар үлесі жоғары, GK толықтығы нөл болуы неліктен мүмкін?
4. Бағалау үшін test-ті алғаш қай сәтте қолдандыңыз және оған дейін не бекітілді?
5. Жолдар орнына клубтар бойынша бөлгенде зерттеу сұрағы қалай өзгереді?

Бағалау, барлығы 100 балл:

- 15 балл — қойылым, схема және белгілер арқылы ағып кетуді болдырмау.
- 15 балл — дұрыс split және индекстерді тексеру.
- 20 балл — базалық модель, Pipeline және түрлендірулерді үйрету.
- 15 балл — тек train ішіндегі CV және нәтижелерді сақтау.
- 15 балл — бекітілген жоспар және бір қорытынды test.
- 10 балл — метрикалар, матрицалар және файлдардың үйлесімділігі.
- 10 балл — түсіндіру, шектеулер және қорғау.

Құжаттама:

- [scikit-learn: деректердің ағып кетуі және жиі қателер](https://scikit-learn.org/stable/common_pitfalls.html).
- [scikit-learn: Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).
- [scikit-learn: метрикалар](https://scikit-learn.org/stable/modules/model_evaluation.html).

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
